# DOES NOT RENAMES 

In [14]:
# -----######-----###### FAST SILENCE-BASED FILE SORTER -----######-----######

import os
import numpy as np
import soundfile as sf
from tqdm import tqdm
import shutil

def _silence_0605fast_sorter_GET_folder_sorted(folder_path, silence_thresh_db=-45, frame_ms=100):
    """
    Analyze AIFF/WAV files and move them into folders based on silence percentage (10%–100%).

    Args:
        folder_path (str): Folder with stems
        silence_thresh_db (float): Silence cutoff in dBFS (default -45)
        frame_ms (int): Frame size in milliseconds (default 100ms)
    """

    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(('.aiff', '.wav')) and not f.startswith('._')]

    print(f"🔍 Processing {len(files)} audio files...\n")

    # Create output folders
    for i in range(10, 110, 10):
        os.makedirs(os.path.join(folder_path, f"{i}_percent_silence"), exist_ok=True)

    for fname in tqdm(files, desc="📦 Sorting by Silence %"):
        fpath = os.path.join(folder_path, fname)

        try:
            data, sr = sf.read(fpath, dtype='float32', always_2d=False)
        except Exception as e:
            print(f"❌ Skipping {fname}: {e}")
            continue

        if data.ndim > 1:  # Stereo → Mono
            data = np.mean(data, axis=1)

        frame_len = int(sr * frame_ms / 1000)
        total_frames = len(data) // frame_len

        if total_frames == 0:
            silence_pct = 100
        else:
            frames = data[:total_frames * frame_len].reshape(total_frames, frame_len)
            rms = np.sqrt(np.mean(frames**2, axis=1))
            db = 20 * np.log10(rms + 1e-10)
            silent = db < silence_thresh_db
            silence_pct = int(np.clip(np.round(np.mean(silent) * 100), 0, 100))

        silence_bin = int(np.ceil(silence_pct / 10.0) * 10)
        silence_bin = max(10, min(100, silence_bin))

        dest_dir = os.path.join(folder_path, f"{silence_bin}_percent_silence")
        shutil.move(fpath, os.path.join(dest_dir, fname))

    print("\n✅ DONE: Files sorted into silence percentage folders.")


In [6]:

folder = "/Volumes/MUSIC_PROD/STEMS_24_years/vocals/__RNRNexports"
_silence_0605fast_sorter_GET_folder_sorted(folder)


# RENAMES 

In [10]:
# -----######-----###### RENAME FILES WITH SILENCE ACRONYM PREFIX -----######-----######

import os
import numpy as np
import soundfile as sf
from tqdm import tqdm

def _silence_0605fast_renamer_GET_prefix_silence(folder_path, silence_thresh_db=-45, frame_ms=100):
    """
    Analyze .aiff/.wav files and rename them with a prefix like SIL30__ based on % of silence.

    Args:
        folder_path (str): Path to audio stems
        silence_thresh_db (float): Threshold for silence in dBFS (default -45)
        frame_ms (int): Frame duration in ms (default 100)
    """

    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(('.aiff', '.wav')) and not f.startswith(('._', 'SIL'))]

    print(f"🔍 Renaming {len(files)} files with silence prefix...\n")

    for fname in tqdm(files, desc="✍️ Prefixing SIL"):
        fpath = os.path.join(folder_path, fname)

        try:
            data, sr = sf.read(fpath, dtype='float32', always_2d=False)
        except Exception as e:
            print(f"❌ Skipping {fname}: {e}")
            continue

        if data.ndim > 1:  # stereo → mono
            data = np.mean(data, axis=1)

        frame_len = int(sr * frame_ms / 1000)
        total_frames = len(data) // frame_len

        if total_frames == 0:
            silence_pct = 100
        else:
            frames = data[:total_frames * frame_len].reshape(total_frames, frame_len)
            rms = np.sqrt(np.mean(frames**2, axis=1))
            db = 20 * np.log10(rms + 1e-10)
            silent = db < silence_thresh_db
            silence_pct = int(np.clip(np.round(np.mean(silent) * 100), 0, 100))

        silence_bin = int(np.ceil(silence_pct / 10.0) * 10)
        silence_bin = max(10, min(100, silence_bin))
        prefix = f"SIL{silence_bin:03d}__"

        new_name = prefix + fname
        new_path = os.path.join(folder_path, new_name)

        if not os.path.exists(new_path):
            os.rename(fpath, new_path)

    print("\n✅ DONE: Files renamed with silence acronym prefix.")


In [11]:

folder = "/Volumes/MUSIC_PROD/STEMS_24_years/_othreSRC_STEM_AIFF/vocals"
_silence_0605fast_renamer_GET_prefix_silence(folder)

🔍 Renaming 1895 files with silence prefix...



✍️ Prefixing SIL: 100%|█████████████████████████████████████████████████| 1895/1895 [2:26:33<00:00,  4.64s/it]


✅ DONE: Files renamed with silence acronym prefix.


In [12]:

folder = "/Volumes/MUSIC_PROD/STEMS_24_years/_othreSRC_STEM_AIFF/drums"
_silence_0605fast_renamer_GET_prefix_silence(folder)

🔍 Renaming 1315 files with silence prefix...



✍️ Prefixing SIL: 100%|█████████████████████████████████████████████████| 1315/1315 [1:02:10<00:00,  2.84s/it]


✅ DONE: Files renamed with silence acronym prefix.


In [13]:

folder = "/Volumes/MUSIC_PROD/STEMS_24_years/_othreSRC_STEM_AIFF/bass"
_silence_0605fast_renamer_GET_prefix_silence(folder)

🔍 Renaming 351 files with silence prefix...



✍️ Prefixing SIL: 100%|█████████████████████████████████████████████████████| 351/351 [27:06<00:00,  4.63s/it]


✅ DONE: Files renamed with silence acronym prefix.
